Knowledge Distillation

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
import torchvision.transforms as transforms
import torchvision.datasets as datasets

In [10]:
device = torch.accelerator.current_accelerator().type if torch.accelerator.is_available() else "cpu"
print(f"Using {device} device")

Using cuda device


Transforming the Dataset

In [11]:
transforms_cifar = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
])

train_dataset = datasets.CIFAR10(root='./data', train=True, download=True, transform=transforms_cifar)
test_dataset = datasets.CIFAR10(root='./data', train=False, download=True, transform=transforms_cifar)

Train/Test Split

In [12]:
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=128, shuffle=True, num_workers=2)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=128, shuffle=False, num_workers=2)

Model Definitions

In [13]:
# Teacher
class DeepNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(128, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.Conv2d(64, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(2048, 512),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(512, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

# Student
class LightNN(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(3, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
            nn.Conv2d(16, 16, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(kernel_size=2, stride=2),
        )
        self.classifier = nn.Sequential(
            nn.Linear(1024, 256),
            nn.ReLU(),
            nn.Dropout(0.1),
            nn.Linear(256, num_classes)
        )

    def forward(self, x):
        x = self.features(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x

Training

In [14]:
def train(model, train_loader, epochs, learning_rate, device):
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    model.train()

    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(inputs)

            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")

def test(model, test_loader, device):
    model.to(device)
    model.eval()

    correct = 0
    total = 0

    with torch.no_grad():
        for inputs, labels in test_loader:
            inputs, labels = inputs.to(device), labels.to(device)

            outputs = model(inputs)
            _, predicted = torch.max(outputs.data, 1)

            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    accuracy = 100 * correct / total
    print(f"Test Accuracy: {accuracy:.2f}%")
    return accuracy

Cross Entropy Runs

Teacher Model Training

In [25]:
torch.manual_seed(42)
nn_deep = DeepNN(num_classes=10).to(device)
train(nn_deep, train_loader, epochs=10, learning_rate=0.001, device=device)
test_accuracy_deep = test(nn_deep, test_loader, device)

Epoch 1/10, Loss: 1.3292563316767172
Epoch 2/10, Loss: 0.8702445752785334
Epoch 3/10, Loss: 0.681902854415157
Epoch 4/10, Loss: 0.5392856876106213
Epoch 5/10, Loss: 0.42335909300143154
Epoch 6/10, Loss: 0.3215206949912069
Epoch 7/10, Loss: 0.23825350062697745
Epoch 8/10, Loss: 0.1708092086226739
Epoch 9/10, Loss: 0.15205806413727344
Epoch 10/10, Loss: 0.12596052419156065
Test Accuracy: 74.84%


In [26]:
torch.manual_seed(42)
nn_light = LightNN(num_classes=10).to(device)
train(nn_light, train_loader, epochs=10, learning_rate=0.001, device=device)
test_accuracy_light = test(nn_light, test_loader, device)

Epoch 1/10, Loss: 1.466608749935999
Epoch 2/10, Loss: 1.1523815318751518
Epoch 3/10, Loss: 1.0226779288952919
Epoch 4/10, Loss: 0.9203796955325719
Epoch 5/10, Loss: 0.8474591531412071
Epoch 6/10, Loss: 0.7801790284683637
Epoch 7/10, Loss: 0.7161195146305787
Epoch 8/10, Loss: 0.6587517552668481
Epoch 9/10, Loss: 0.6053066725468697
Epoch 10/10, Loss: 0.5542577972345035
Test Accuracy: 70.19%


In [27]:
torch.manual_seed(42)
nn_light2 = LightNN(num_classes=10).to(device)
train(nn_light2, train_loader, epochs=10, learning_rate=0.001, device=device)
test_accuracy_light2 = test(nn_light2, test_loader, device)

Epoch 1/10, Loss: 1.4709458021861512
Epoch 2/10, Loss: 1.1579751155870346
Epoch 3/10, Loss: 1.0213370861299813
Epoch 4/10, Loss: 0.9203909899267699
Epoch 5/10, Loss: 0.8462806020856208
Epoch 6/10, Loss: 0.7797329537094096
Epoch 7/10, Loss: 0.7149361690596852
Epoch 8/10, Loss: 0.658395433593589
Epoch 9/10, Loss: 0.6038044848862816
Epoch 10/10, Loss: 0.5545906980171837
Test Accuracy: 70.53%


In [22]:
print(f"Teacher accuracy: {test_accuracy_deep:.2f}%")
print(f"Student accuracy: {test_accuracy_light:.2f}%")
print(f"Student accuracy: {test_accuracy_light2:.2f}%")

Teacher accuracy: 75.37%
Student accuracy: 69.86%
Student accuracy: 70.48%


Knowledge Distillation

In [23]:
def train_knowledge_distillation(teacher, student, train_loader, epochs, learning_rate, T, soft_target_loss_weight, ce_loss_weight, device):
    ce_loss = nn.CrossEntropyLoss()
    optimizer = optim.Adam(student.parameters(), lr=learning_rate)

    teacher.eval()
    student.train()

    for epoch in range(epochs):
        running_loss = 0.0
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(device), labels.to(device)
            optimizer.zero_grad()
            with torch.no_grad():
                teacher_logits = teacher(inputs)

            student_logits = student(inputs)

            soft_targets = nn.functional.softmax(teacher_logits / T, dim=-1)
            soft_prob = nn.functional.log_softmax(student_logits / T, dim=-1)
            soft_targets_loss = torch.sum(soft_targets * (soft_targets.log() - soft_prob)) / soft_prob.size()[0] * (T**2)

            label_loss = ce_loss(student_logits, labels)
            loss = soft_target_loss_weight * soft_targets_loss + ce_loss_weight * label_loss

            loss.backward()
            optimizer.step()
            running_loss += loss.item()

        print(f"Epoch {epoch+1}/{epochs}, Loss: {running_loss / len(train_loader)}")

In [29]:
train_knowledge_distillation(teacher=nn_deep, student=nn_light, train_loader=train_loader, epochs=20, learning_rate=0.001, T=2, soft_target_loss_weight=0.25, ce_loss_weight=0.75, device=device)
test_accuracy_light_ce_and_kd = test(nn_light, test_loader, device)

print(f"Teacher accuracy: {test_accuracy_deep:.2f}%")
print(f"Student accuracy without teacher: {test_accuracy_light2:.2f}%")
print(f"Student accuracy with CE + KD: {test_accuracy_light_ce_and_kd:.2f}%")

Epoch 1/20, Loss: 0.42654717380128554
Epoch 2/20, Loss: 0.3937828540802002
Epoch 3/20, Loss: 0.38522905747756325
Epoch 4/20, Loss: 0.3605635655886682
Epoch 5/20, Loss: 0.3494672473434292
Epoch 6/20, Loss: 0.341717044296472
Epoch 7/20, Loss: 0.3252825741572758
Epoch 8/20, Loss: 0.3142521838702814
Epoch 9/20, Loss: 0.30905465907452967
Epoch 10/20, Loss: 0.2991836941455636
Epoch 11/20, Loss: 0.29089344515824866
Epoch 12/20, Loss: 0.2854025891750975
Epoch 13/20, Loss: 0.27753277713685387
Epoch 14/20, Loss: 0.2776291941666542
Epoch 15/20, Loss: 0.26933684041890343
Epoch 16/20, Loss: 0.26740597912570097
Epoch 17/20, Loss: 0.26021968441851
Epoch 18/20, Loss: 0.2524467167418326
Epoch 19/20, Loss: 0.24502337752553202
Epoch 20/20, Loss: 0.25085473860926033
Test Accuracy: 69.58%
Teacher accuracy: 74.84%
Student accuracy without teacher: 70.53%
Student accuracy with CE + KD: 69.58%
